# Transfer Trajectory: UKR+COVID+Midterm NM Aug

Visualizes transfer evals for `merged_ukr_rus_covid_midterm_nm_aug_15_06_2026_17_49_22` at checkpoints 1k, 10k, 50k, and 100k.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

CSV_PATH = Path("eval_results.csv")
if not CSV_PATH.exists():
    CSV_PATH = Path("scripts/plotting/transfer_trajectory_merged_ukr_rus_covid_midterm_nm_aug/eval_results.csv")
CSV_PATH = CSV_PATH.resolve()

SPLIT = "test"
METRIC = "roc_auc"
METRICS = ["accuracy", "f1", "roc_auc"]
REQUESTED_STEPS = [1000, 10000, 50000, 100000]
SHOT_ORDER = [0, 3, 10]
DATASET_ORDER = [
    "midterm",
    "covid19_twitter",
    "ukr_rus_twitter",
    "covid_political",
    "election2020",
    "ukr_rus_suspended",
]
TASK_ORDER = ["nm", "lp", "pl"]
TASK_LABELS = {
    "nm": "Neighbor matching",
    "lp": "Temporal link prediction",
    "pl": "Classification",
}
METRIC_LABELS = {
    "accuracy": "Accuracy",
    "f1": "F1",
    "roc_auc": "ROC-AUC",
}
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

In [ ]:
df = pd.read_csv(CSV_PATH)
required = {"model", "dataset", "task", "shots", "split", *METRICS}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"CSV is missing required columns: {sorted(missing)}")

df["shots"] = pd.to_numeric(df["shots"], errors="raise").astype(int)
for col in METRICS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["step"] = df["model"].str.extract(r"_step([0-9]+)$", expand=False)
if df["step"].isna().any():
    bad = df.loc[df["step"].isna(), "model"].tolist()
    raise ValueError(f"Could not parse checkpoint step from model names: {bad}")
df["step"] = df["step"].astype(int)

df = df[(df["split"] == SPLIT) & (df["step"].isin(REQUESTED_STEPS))].copy()
df["dataset"] = pd.Categorical(df["dataset"], categories=DATASET_ORDER, ordered=True)
df["task"] = pd.Categorical(df["task"], categories=TASK_ORDER, ordered=True)
df["shots"] = pd.Categorical(df["shots"], categories=SHOT_ORDER, ordered=True)
df = df.sort_values(["dataset", "task", "shots", "step"])

present_steps = sorted(df["step"].dropna().unique().tolist())
missing_steps = sorted(set(REQUESTED_STEPS) - set(present_steps))
print(f"reading {CSV_PATH}")
print(f"rows: {len(df)}")
print(f"present steps: {present_steps}")
print(f"missing requested steps: {missing_steps}")

coverage = (
    df.groupby(["dataset", "task"], observed=True)
      .agg(
          shots=("shots", lambda x: sorted(set(int(v) for v in x.dropna()))),
          steps=("step", lambda x: sorted(set(int(v) for v in x.dropna()))),
          rows=("step", "size"),
      )
      .reset_index()
      .sort_values(["dataset", "task"])
)
display(coverage)

In [ ]:
def plot_metric_grid(data, metric=METRIC):
    subset = data[data[metric].notna()].copy()
    if subset.empty:
        raise ValueError(f"No non-null values for metric={metric!r}")

    datasets = [d for d in DATASET_ORDER if d in set(subset["dataset"].astype(str))]
    tasks = [t for t in TASK_ORDER if t in set(subset["task"].astype(str))]
    palette = dict(zip(SHOT_ORDER, sns.color_palette("Set2", n_colors=len(SHOT_ORDER))))

    fig, axes = plt.subplots(
        nrows=len(datasets),
        ncols=len(tasks),
        figsize=(4.7 * len(tasks), 2.5 * len(datasets)),
        sharex=True,
        sharey=True,
    )
    if len(datasets) == 1 and len(tasks) == 1:
        axes = [[axes]]
    elif len(datasets) == 1:
        axes = [axes]
    elif len(tasks) == 1:
        axes = [[ax] for ax in axes]

    for row_idx, dataset in enumerate(datasets):
        for col_idx, task in enumerate(tasks):
            ax = axes[row_idx][col_idx]
            panel = subset[(subset["dataset"].astype(str) == dataset) & (subset["task"].astype(str) == task)]
            ax.set_title(f"{dataset}\n{TASK_LABELS.get(task, task)}", fontsize=10)
            if panel.empty:
                ax.text(0.5, 0.5, "no data", transform=ax.transAxes, ha="center", va="center", color="0.45")
            else:
                for shot in SHOT_ORDER:
                    line = panel[panel["shots"].astype(str) == str(shot)].sort_values("step")
                    if line.empty:
                        continue
                    ax.plot(
                        line["step"],
                        line[metric],
                        marker="o",
                        linewidth=2,
                        markersize=4,
                        color=palette[shot],
                        label=f"{shot}-shot",
                    )
            ax.set_xticks(REQUESTED_STEPS)
            ax.set_xticklabels(["1k", "10k", "50k", "100k"], rotation=30, ha="right")
            ax.set_ylim(0, 1.02)
            ax.grid(True, alpha=0.3)
            if row_idx == len(datasets) - 1:
                ax.set_xlabel("checkpoint")
            if col_idx == 0:
                ax.set_ylabel(METRIC_LABELS.get(metric, metric))

    handles, labels = axes[0][0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, title="shots", loc="upper center", ncol=len(labels), bbox_to_anchor=(0.5, 1.02))
    fig.suptitle(f"Transfer trajectory by checkpoint - {METRIC_LABELS.get(metric, metric)}", y=1.04, fontsize=14)
    fig.tight_layout()
    out_path = FIG_DIR / f"transfer_trajectory_{metric}.png"
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    return fig

plot_metric_grid(df, METRIC)
plt.show()

In [ ]:
for metric in METRICS:
    plot_metric_grid(df, metric)
    plt.show()

In [ ]:
best_rows = []
for metric in METRICS:
    valid = df[df[metric].notna()]
    if valid.empty:
        continue
    idx = valid.groupby(["dataset", "task", "shots"], observed=True)[metric].idxmax()
    best = valid.loc[idx, ["dataset", "task", "shots", "step", metric]].copy()
    best["metric"] = metric
    best = best.rename(columns={metric: "value", "step": "best_step"})
    best_rows.append(best)

best_table = pd.concat(best_rows, ignore_index=True) if best_rows else pd.DataFrame()
best_table["metric_label"] = best_table["metric"].map(METRIC_LABELS)
best_table = best_table.sort_values(["dataset", "task", "shots", "metric"])
display(best_table[["dataset", "task", "shots", "metric_label", "best_step", "value"]].style.format({"value": "{:.4f}"}))

In [ ]:
wide = (
    df.pivot_table(
        index=["dataset", "task", "shots"],
        columns="step",
        values=METRIC,
        observed=True,
    )
    .reindex(columns=REQUESTED_STEPS)
    .sort_index()
)
display(wide.style.format("{:.4f}").background_gradient(cmap="YlGnBu", axis=1))